In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
import cv2
import pydicom
from datetime import datetime
import json
import requests
from PIL import Image

# 1. First, define the model architecture exactly as used in training
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        # Use latest syntax for loading pretrained models
        self.vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        for param in self.vgg16.features.parameters():
            param.requires_grad = False
        in_feats = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(in_feats, 2)
    
    def forward(self, x):
        x = self.vgg16(x)
        return x

# 2. Define the dataset class (needed for reference)
class BrainCTDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            if img_path.lower().endswith('.dcm'):
                dcm = pydicom.dcmread(img_path)
                image = dcm.pixel_array.astype(float)
                window_center = 40
                window_width = 80
                min_value = window_center - window_width // 2
                max_value = window_center + window_width // 2
                image = np.clip(image, min_value, max_value)
                if len(image.shape) == 2:
                    image = np.stack([image] * 3, axis=-1)
            else:
                image = cv2.imread(img_path)
                if image is None:
                    raise ValueError(f"Failed to load image: {img_path}")
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-7)
            image = image.astype(np.float32)
            if self.transform:
                image = self.transform(torch.from_numpy(image).permute(2, 0, 1))
            return image, label
        except Exception as e:
            print(f"Error processing image {img_path}: {str(e)}")
            return None, None

# Modified load_model_safe function
def load_model_safe(model_path, device='cpu'):
    """Load model with enhanced error handling and PyTorch 2.6+ compatibility"""
    print(f"\nLoading model at {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"User: krooldonutz")
    
    try:
        # Initialize model
        model = CNNModel()
        
        # Method 1: Try loading with safe globals
        try:
            from torch.serialization import safe_globals, add_safe_globals
            add_safe_globals([torch.torch_version.TorchVersion])
            with safe_globals([torch.torch_version.TorchVersion]):
                checkpoint = torch.load(model_path, map_location=device)
            print("✓ Loaded with safe globals")
        except Exception as e1:
            print(f"Safe globals loading failed: {str(e1)}")
            try:
                # Method 2: Try loading with weights_only=False
                checkpoint = torch.load(model_path, map_location=device, weights_only=False)
                print("✓ Loaded with weights_only=False")
            except Exception as e2:
                print(f"Standard loading failed: {str(e2)}")
                try:
                    # Method 3: Try direct pickle loading
                    import pickle
                    with open(model_path, 'rb') as f:
                        checkpoint = pickle.load(f)
                    print("✓ Loaded with pickle")
                except Exception as e3:
                    print("All loading methods failed:")
                    print(f"- Safe globals error: {str(e1)}")
                    print(f"- Standard loading error: {str(e2)}")
                    print(f"- Pickle loading error: {str(e3)}")
                    raise Exception("Failed to load model with any method")
        
        # Handle different checkpoint formats
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                state_dict = checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
            else:
                state_dict = checkpoint
        else:
            state_dict = checkpoint
        
        # Load state dict with strict=False to handle missing keys
        incompatible_keys = model.load_state_dict(state_dict, strict=False)
        
        # Print warnings for incompatible keys
        if incompatible_keys.missing_keys:
            print("\nWarning - Missing keys:")
            for key in incompatible_keys.missing_keys:
                print(f"- {key}")
        if incompatible_keys.unexpected_keys:
            print("\nWarning - Unexpected keys:")
            for key in incompatible_keys.unexpected_keys:
                print(f"- {key}")
        
        model = model.to(device)
        model.eval()
        print("✓ Model initialized and set to evaluation mode")
        return model
        
    except Exception as e:
        print(f"❌ Error loading model: {str(e)}")
        raise


# 4. Define preprocessing function (similar to dataset class)
def preprocess_image(image_path_or_url):
    """Preprocess image for inference"""
    try:
        # Handle URL or local path
        if image_path_or_url.startswith('http'):
            response = requests.get(image_path_or_url)
            image_array = np.asarray(bytearray(response.content), dtype=np.uint8)
            image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
        else:
            image = cv2.imread(image_path_or_url)
        
        if image is None:
            raise ValueError("Failed to load image")
            
        # Convert BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Normalize
        image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-7)
        image = image.astype(np.float32)
        
        # Apply transforms
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
        
        image_tensor = transform(image)
        return image_tensor.unsqueeze(0)
        
    except Exception as e:
        print(f"❌ Error preprocessing image: {str(e)}")
        raise

# 5. Define inference function
def predict(model, image_path_or_url, device='cpu'):
    """Run inference on an image"""
    try:
        # Preprocess
        input_tensor = preprocess_image(image_path_or_url)
        input_tensor = input_tensor.to(device)
        
        # Predict
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = torch.nn.functional.softmax(output, dim=1)
            
        # Get prediction
        predictions = probabilities[0].cpu().numpy()
        prediction = "Aneurysm" if predictions[1] > predictions[0] else "Non-aneurysm"
        confidence = float(max(predictions))
        
        return {
            "prediction": prediction,
            "confidence": confidence,
            "probabilities": {
                "non_aneurysm": float(predictions[0]),
                "aneurysm": float(predictions[1])
            },
            "timestamp": datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'),
            "user": "krooldonutz"
        }
        
    except Exception as e:
        print(f"❌ Error during prediction: {str(e)}")
        raise

# Modified test_inference_pipeline function
def test_inference_pipeline():
    """Test the complete inference pipeline with enhanced error handling"""
    try:
        print(f"\n{'='*50}")
        print(f"Starting test at {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
        print(f"User: krooldonutz")
        print(f"PyTorch version: {torch.__version__}")
        
        # Setup
        model_dir = 'my_model'
        model_path = os.path.join(model_dir, 'model.pth')
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Verify model file
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model not found at: {model_path}")
        print(f"Model file size: {os.path.getsize(model_path)/1024/1024:.2f}MB")
        
        # Load model
        model = load_model_safe(model_path, device)
        
        # Test image
        test_url = "https://img.medscapestatic.com/pi/meds/ckb/35/15035tn.jpg"
        
        # Run prediction
        print("\nRunning prediction...")
        result = predict(model, test_url, device)
        
        # Print results
        print("\nPrediction Results:")
        print(json.dumps(result, indent=2))
        
        print(f"\n{'='*50}")
        print("TEST COMPLETED SUCCESSFULLY")
        print(f"{'='*50}")
        
    except Exception as e:
        print(f"\n{'='*50}")
        print("TEST FAILED")
        print(f"{'='*50}")
        print(f"Error: {str(e)}")
        import traceback
        print("\nTraceback:")
        print(traceback.format_exc())
    finally:
        # Cleanup
        if 'model' in locals():
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Add this function to handle model conversion if needed
def convert_model_format(old_path, new_path=None):
    """Convert model to new format if needed"""
    if new_path is None:
        new_path = old_path + '.converted'
    
    try:
        print(f"\nAttempting to convert model at {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
        
        # Load old model
        checkpoint = torch.load(old_path, map_location='cpu', weights_only=False)
        
        # Extract state dict
        if isinstance(checkpoint, dict):
            state_dict = checkpoint.get('model_state_dict', 
                        checkpoint.get('state_dict', checkpoint))
        else:
            state_dict = checkpoint
        
        # Save in new format
        torch.save({
            'model_state_dict': state_dict,
            'pytorch_version': torch.__version__,
            'conversion_date': datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'),
            'converted_by': 'krooldonutz'
        }, new_path, _use_new_zipfile_serialization=False)
        
        print(f"✓ Model converted and saved to: {new_path}")
        return new_path
        
    except Exception as e:
        print(f"❌ Conversion failed: {str(e)}")
        raise

if __name__ == "__main__":
    try:
        # Try adding safe globals first
        from torch.serialization import add_safe_globals
        add_safe_globals([torch.torch_version.TorchVersion])
    except Exception as e:
        print(f"Warning: Could not add safe globals: {str(e)}")
    
    # Run the test
    test_inference_pipeline()

In [ ]:
def verify_state_dict(state_dict):
    """Verify the structure of the state dict"""
    print("\nVerifying state dict structure...")
    
    expected_keys = {
        'vgg16.0.weight', 'vgg16.0.bias',
        'vgg16.2.weight', 'vgg16.2.bias',
        'vgg16.5.weight', 'vgg16.5.bias',
        'vgg16.7.weight', 'vgg16.7.bias',
        'vgg16.10.weight', 'vgg16.10.bias',
        'vgg16.12.weight', 'vgg16.12.bias',
        'classifier.0.weight', 'classifier.0.bias',
        'classifier.3.weight', 'classifier.3.bias'
    }
    
    actual_keys = set(state_dict.keys())
    
    print("\nState dict contents:")
    for key in actual_keys:
        if isinstance(state_dict[key], torch.Tensor):
            print(f"- {key}: {state_dict[key].shape}")
    
    missing_keys = expected_keys - actual_keys
    unexpected_keys = actual_keys - expected_keys
    
    if missing_keys:
        print("\nMissing keys:")
        for key in missing_keys:
            print(f"- {key}")
    
    if unexpected_keys:
        print("\nUnexpected keys:")
        for key in unexpected_keys:
            print(f"- {key}")
            
    return len(missing_keys) == 0

verify

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms, models
import requests
import cv2
import torch
import torch.nn as nn
import traceback
from datetime import datetime

def add_safe_globals(safe_list):
    """Helper function to add safe globals for model loading"""
    return safe_list

class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        # Initialize VGG16 with pretrained weights
        self.vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

        # Freeze feature layers
        for param in self.vgg16.features.parameters():
            param.requires_grad = False

        # Modify the classifier
        in_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(in_features, 2)

    def forward(self, x):
        return self.vgg16(x)

def load_model_safely(model_path, device):
    print(f"\nLoading model at {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"User: krooldonutz")
    print(f"PyTorch version: {torch.__version__}")

    try:
        # Load with weights_only=False and safe globals
        add_safe_globals([torch.torch_version.TorchVersion])
        checkpoint = torch.load(model_path, weights_only=False, map_location=device)

        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                return checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint:
                return checkpoint['state_dict']
            else:
                return checkpoint
        return checkpoint
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        raise

def create_multiclass_shap_values(model_path, image_url):
    print(f"Analysis started at {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"Analysis requested by: krooldonutz")

    try:
        # Set up device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Initialize model
        model = CNNModel().to(device)
        
        # Load state dict
        state_dict = load_model_safely(model_path, device)
        model.load_state_dict(state_dict)
        model.eval()

        # Download and preprocess image
        response = requests.get(image_url)
        if response.status_code != 200:
            return "Error: Could not download image from URL"

        image_array = np.asarray(bytearray(response.content), dtype=np.uint8)
        image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Resize image to 224x224
        image = cv2.resize(image, (224, 224))

        # Preprocess image
        image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-7)
        image = image.astype(np.float32)

        # Create masker that is used to mask out parts of the image
        masker = shap.maskers.Image("inpaint_telea", (224, 224, 3))

        # Create a wrapper for the model that handles preprocessing
        def model_pipeline(x):
            # Convert to torch tensor and move channels first
            x = torch.Tensor(x).permute(0, 3, 1, 2)

            # Apply normalization
            normalize = transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
            x = torch.stack([normalize(img) for img in x])

            # Get model predictions
            with torch.no_grad():
                x = x.to(device)
                output = model(x)
                probs = torch.nn.functional.softmax(output, dim=1)
            return probs.cpu().numpy()

        # Create explainer
        explainer = shap.Explainer(
            model=model_pipeline,
            masker=masker,
            output_names=['Non-aneurysm', 'Aneurysm']
        )

        # Create a batch with the same image repeated twice
        image_batch = np.stack([image, image])  # Create batch of size 2

        print("Calculating SHAP values...")
        # Get SHAP values
        shap_values = explainer(image_batch, max_evals=100, batch_size=50)

        return shap_values

    except Exception as e:
        print(f"\nError occurred at {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
        return f"Error generating SHAP values: {str(e)}", traceback.format_exc()

# Example usage:
if __name__ == "__main__":
    model_path = "my_model/model.pth"
    image_url = "https://prod-images-static.radiopaedia.org/images/61513769/9131de5120872bcc1ad19d8198c759904a4b78efa405550c86ec0f19d3dda590_big_gallery.jpeg"
    
    try:
        shap_values = create_multiclass_shap_values(model_path, image_url)
        print(shap_values)
        
    except Exception as e:
        print(f"Error in main execution: {str(e)}")
